# EP20. 에이전트 메모리 아키텍처

## 학습 목표

1. **Short-term / Long-term 메모리**의 차이를 이해하고, 각각의 구현 방식을 실습한다
2. **3가지 저장 전략** (Vector DB / KV Store / Markdown)의 장단점을 비교하고 직접 구현한다
3. **메모리 압축과 검색 전략** (Recency + Relevance + Importance)을 적용하여 토큰을 절약한다
4. **LangGraph Checkpointer**와 **Langfuse**를 통합하여 세션 간 기억이 유지되는 프로덕션 에이전트를 구축한다

---

### AI 가이드

이 노트북은 에이전트에 장기 기억을 부여하는 핵심 아키텍처를 다룹니다.
기억 못 하는 에이전트는 매 세션마다 처음부터 시작합니다 — 이 문제를 해결합니다.

### 사전 요구사항

- Python 3.10+
- Anthropic API 키 (Claude)
- Langfuse 계정 (무료 클라우드 또는 셀프호스팅)

### 예상 소요 시간: 70분

### API 키 필요

- `ANTHROPIC_API_KEY`: Claude API 호출
- `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY`: Langfuse 트레이싱

---
## 섹션 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
!uv pip install langgraph langchain langchain-anthropic langfuse chromadb sentence-transformers tiktoken python-dotenv --quiet

---
## 섹션 2. 라이브러리 로드 + API 키 + Langfuse 초기화

In [ ]:
import os
import json
import math
import warnings
from datetime import datetime, timedelta
from uuid import uuid4
from typing import Optional

import tiktoken
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

# API 키 설정 — .env 파일에 저장하거나 아래에 직접 입력
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"

print("ANTHROPIC_API_KEY:", "설정됨" if os.environ.get("ANTHROPIC_API_KEY") else "미설정")
print("LANGFUSE_PUBLIC_KEY:", "설정됨" if os.environ.get("LANGFUSE_PUBLIC_KEY") else "미설정")

In [ ]:
# Langfuse 초기화
from langfuse import Langfuse

langfuse = Langfuse()
print("Langfuse 연결 성공:", langfuse.auth_check())

In [ ]:
# 토큰 카운팅 유틸리티
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    """텍스트의 토큰 수를 반환합니다."""
    return len(enc.encode(text))

# 테스트
sample = "에이전트 메모리 아키텍처를 배워봅시다."
print(f"'{sample}' → {count_tokens(sample)} 토큰")

---
## 섹션 3. Short-term Memory 구현

Short-term Memory는 현재 세션의 대화 기록을 관리합니다.
토큰 제한을 초과하지 않도록 **슬라이딩 윈도우** 방식을 적용합니다.

In [ ]:
from langchain.memory import ConversationBufferMemory, ConversationBufferWindowMemory

# 1) 기본 버퍼 메모리 — 모든 대화를 저장 (토큰 증가 문제)
buffer_memory = ConversationBufferMemory(return_messages=True)

# 대화 시뮬레이션
conversations = [
    ("나는 김철수야, ML 엔지니어로 일하고 있어", "안녕하세요 김철수님! ML 엔지니어시군요."),
    ("요즘 RAG 파이프라인을 개선하고 있어", "RAG 파이프라인 개선 작업이시군요. 어떤 부분을 개선하시나요?"),
    ("리랭킹 모델을 추가하려고 해", "리랭킹 추가는 좋은 선택입니다. Cross-encoder를 추천합니다."),
    ("Python으로 구현하고 싶은데 라이브러리 추천해줘", "sentence-transformers의 CrossEncoder를 추천합니다."),
    ("성능 벤치마크 방법도 알려줘", "NDCG, MRR, Recall@k 등의 메트릭을 사용하세요."),
]

for human, ai in conversations:
    buffer_memory.save_context({"input": human}, {"output": ai})

all_messages = buffer_memory.load_memory_variables({})["history"]
full_text = "\n".join([m.content for m in all_messages])
print(f"전체 버퍼 메모리: {len(all_messages)}개 메시지, {count_tokens(full_text)} 토큰")
print(f"\n--- 전체 대화 ---")
for msg in all_messages:
    role = "사용자" if msg.type == "human" else "AI"
    print(f"[{role}] {msg.content}")

In [ ]:
# 2) 슬라이딩 윈도우 메모리 — 최근 N턴만 유지
window_memory = ConversationBufferWindowMemory(k=3, return_messages=True)

for human, ai in conversations:
    window_memory.save_context({"input": human}, {"output": ai})

window_messages = window_memory.load_memory_variables({})["history"]
window_text = "\n".join([m.content for m in window_messages])
print(f"윈도우 메모리 (k=3): {len(window_messages)}개 메시지, {count_tokens(window_text)} 토큰")
print(f"\n--- 최근 3턴 ---")
for msg in window_messages:
    role = "사용자" if msg.type == "human" else "AI"
    print(f"[{role}] {msg.content}")

print(f"\n토큰 절약: {count_tokens(full_text)} → {count_tokens(window_text)} "
      f"({100 - count_tokens(window_text) * 100 // count_tokens(full_text)}% 절약)")

---
## 섹션 4. Long-term Memory — Vector DB 방식 (ChromaDB)

의미 기반 검색이 가능한 장기 메모리입니다.
대화에서 추출한 기억을 임베딩으로 저장하고, 유사도 검색으로 불러옵니다.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# 임베딩 모델 로드
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"임베딩 모델 로드 완료: all-MiniLM-L6-v2 (차원: {embedder.get_sentence_embedding_dimension()})")

# ChromaDB 클라이언트 생성 (인메모리)
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="agent_memory",
    metadata={"hnsw:space": "cosine"}
)
print(f"ChromaDB 컬렉션 생성: agent_memory")

In [ ]:
# 메모리 저장 함수
def store_memory_vectordb(content: str, memory_type: str, importance: int = 5):
    """메모리를 Vector DB에 저장합니다.
    
    Args:
        content: 기억 내용
        memory_type: episodic / semantic / procedural
        importance: 중요도 1~10
    """
    embedding = embedder.encode(content).tolist()
    mem_id = f"mem_{uuid4().hex[:8]}"
    collection.add(
        documents=[content],
        embeddings=[embedding],
        metadatas=[{
            "type": memory_type,
            "importance": importance,
            "timestamp": datetime.now().isoformat(),
        }],
        ids=[mem_id]
    )
    return mem_id

# 메모리 검색 함수
def recall_from_vectordb(query: str, k: int = 3):
    """쿼리와 관련된 메모리를 검색합니다."""
    query_embedding = embedder.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    return results

# 기억 저장
memories = [
    ("사용자 김철수는 ML 엔지니어이다", "semantic", 9),
    ("김철수가 RAG 파이프라인 개선 프로젝트를 진행 중이다", "episodic", 7),
    ("리랭킹 모델로 Cross-encoder를 추천받았다", "episodic", 6),
    ("사용자는 Python을 주력 언어로 사용한다", "semantic", 8),
    ("코드에 타입힌트를 항상 포함하는 것을 선호한다", "procedural", 7),
    ("sentence-transformers 라이브러리 사용 경험이 있다", "semantic", 5),
    ("벤치마크 메트릭으로 NDCG, MRR, Recall@k를 사용하기로 했다", "episodic", 6),
    ("비동기 코드 작성을 선호한다", "procedural", 6),
]

for content, mtype, imp in memories:
    mid = store_memory_vectordb(content, mtype, imp)
    print(f"  저장: [{mtype:10s}] (중요도:{imp}) {content} → {mid}")

print(f"\n총 {collection.count()}개 메모리 저장 완료")

In [ ]:
# 메모리 검색 테스트
test_queries = [
    "사용자의 직업은?",
    "어떤 프로젝트를 진행 중인가?",
    "코딩 스타일 선호도는?",
]

for query in test_queries:
    print(f"\n질문: {query}")
    results = recall_from_vectordb(query, k=3)
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    )):
        sim = 1 - dist  # cosine distance → similarity
        print(f"  {i+1}. [{meta['type']:10s}] (유사도: {sim:.3f}) {doc}")

---
## 섹션 5. Long-term Memory — KV Store 방식

키-값 기반으로 사용자 프로필이나 설정을 빠르게 조회합니다.
의미 검색은 불가능하지만, 정확한 키로 즉시 접근할 수 있습니다.

In [ ]:
class KVMemory:
    """키-값 기반 메모리 저장소.
    
    사용자 프로필, 설정, 자주 조회되는 정보를 저장합니다.
    실제 프로덕션에서는 Redis 등으로 대체합니다.
    """
    def __init__(self):
        self.store: dict[str, dict] = {}
    
    def set(self, key: str, value: str, category: str = "general"):
        """키-값 쌍을 저장합니다."""
        self.store[key] = {
            "value": value,
            "category": category,
            "updated_at": datetime.now().isoformat(),
        }
    
    def get(self, key: str) -> Optional[str]:
        """키로 값을 조회합니다."""
        entry = self.store.get(key)
        return entry["value"] if entry else None
    
    def get_by_category(self, category: str) -> dict[str, str]:
        """카테고리별 모든 항목을 반환합니다."""
        return {
            k: v["value"] 
            for k, v in self.store.items() 
            if v["category"] == category
        }
    
    def delete(self, key: str) -> bool:
        """키를 삭제합니다."""
        if key in self.store:
            del self.store[key]
            return True
        return False
    
    def list_keys(self) -> list[str]:
        """모든 키 목록을 반환합니다."""
        return list(self.store.keys())
    
    def to_context_string(self) -> str:
        """시스템 프롬프트에 주입할 수 있는 문자열로 변환합니다."""
        lines = []
        for key, entry in self.store.items():
            lines.append(f"- {key}: {entry['value']}")
        return "\n".join(lines)


# KV 메모리 사용
kv = KVMemory()

# 사용자 프로필 저장
kv.set("user_name", "김철수", category="profile")
kv.set("user_job", "ML 엔지니어", category="profile")
kv.set("preferred_language", "Python", category="preference")
kv.set("coding_style", "타입힌트 포함, 비동기 우선", category="preference")
kv.set("current_project", "RAG 파이프라인 개선", category="context")

# 조회 테스트
print("=== 개별 조회 ===")
print(f"사용자 이름: {kv.get('user_name')}")
print(f"현재 프로젝트: {kv.get('current_project')}")

print(f"\n=== 카테고리별 조회 ===")
print(f"프로필: {kv.get_by_category('profile')}")
print(f"선호도: {kv.get_by_category('preference')}")

print(f"\n=== 컨텍스트 문자열 ({count_tokens(kv.to_context_string())} 토큰) ===")
print(kv.to_context_string())

---
## 섹션 6. Long-term Memory — Markdown 파일 방식

사람이 읽고 편집할 수 있는 마크다운 파일에 기억을 저장합니다.
Claude의 `CLAUDE.md`나 Cursor의 `.cursorrules`와 유사한 접근입니다.

In [ ]:
import tempfile
import pathlib

class MarkdownMemory:
    """마크다운 파일 기반 메모리 저장소.
    
    사람이 읽고 편집할 수 있으며, Git으로 버전 관리가 가능합니다.
    """
    def __init__(self, file_path: str):
        self.file_path = pathlib.Path(file_path)
        if not self.file_path.exists():
            self.file_path.write_text("# Agent Memory\n\n", encoding="utf-8")
    
    def add(self, category: str, content: str):
        """카테고리별로 기억을 추가합니다."""
        current = self.file_path.read_text(encoding="utf-8")
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
        
        # 카테고리 섹션이 있으면 추가, 없으면 새로 생성
        section_header = f"## {category}"
        entry = f"- {content} _{timestamp}_"
        
        if section_header in current:
            current = current.replace(
                section_header,
                f"{section_header}\n{entry}"
            )
        else:
            current += f"\n{section_header}\n{entry}\n"
        
        self.file_path.write_text(current, encoding="utf-8")
    
    def load(self) -> str:
        """전체 메모리 파일을 읽어옵니다."""
        return self.file_path.read_text(encoding="utf-8")
    
    def get_section(self, category: str) -> list[str]:
        """특정 카테고리의 기억만 반환합니다."""
        content = self.load()
        lines = content.split("\n")
        in_section = False
        entries = []
        for line in lines:
            if line.strip() == f"## {category}":
                in_section = True
                continue
            elif line.startswith("## ") and in_section:
                break
            elif in_section and line.strip().startswith("- "):
                entries.append(line.strip()[2:])
        return entries


# 임시 파일로 테스트
tmp_dir = tempfile.mkdtemp()
md_mem = MarkdownMemory(f"{tmp_dir}/agent_memory.md")

# 기억 저장
md_mem.add("사용자 정보", "이름: 김철수, ML 엔지니어")
md_mem.add("사용자 정보", "Python 주력, 타입힌트 선호")
md_mem.add("프로젝트", "RAG 파이프라인 개선 진행 중")
md_mem.add("프로젝트", "리랭킹 모델 Cross-encoder 도입 예정")
md_mem.add("선호도", "비동기 코드 선호, 간결한 설명 요청")

# 전체 읽기
full_md = md_mem.load()
print(f"=== Markdown 메모리 ({count_tokens(full_md)} 토큰) ===")
print(full_md)

# 특정 카테고리 조회
print(f"\n=== '프로젝트' 섹션 ===")
for entry in md_mem.get_section("프로젝트"):
    print(f"  - {entry}")

---
## 섹션 7. 메모리 압축

오래된 대화를 LLM으로 요약하여 토큰을 절약합니다.
압축 전/후 토큰 수를 비교해봅니다.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)

# 압축할 긴 대화 기록
long_conversation = """사용자: 안녕하세요, 저는 김철수입니다. ML 엔지니어로 일하고 있어요.
AI: 안녕하세요 김철수님! ML 엔지니어시군요. 무엇을 도와드릴까요?
사용자: 요즘 RAG 파이프라인을 개선하고 있는데, 검색 품질이 안 좋아서 고민이에요.
AI: RAG 검색 품질 개선에는 여러 방법이 있습니다. 어떤 부분이 가장 문제인가요?
사용자: 관련 없는 문서가 너무 많이 검색되는 게 문제입니다.
AI: 그렇다면 리랭킹(reranking) 모델을 도입하는 것을 추천합니다. Cross-encoder 기반의 리랭커가 효과적입니다.
사용자: Cross-encoder가 뭔가요? Bi-encoder와 차이가 뭐죠?
AI: Bi-encoder는 쿼리와 문서를 각각 임베딩하여 비교하고, Cross-encoder는 쿼리와 문서를 함께 입력받아 직접 관련도를 판단합니다. Cross-encoder가 더 정확하지만 느립니다.
사용자: 그러면 실제 코드로 어떻게 구현하나요? Python으로 하고 싶어요.
AI: sentence-transformers 라이브러리의 CrossEncoder 클래스를 사용하면 됩니다. 먼저 pip install sentence-transformers를 실행하세요.
사용자: 타입힌트도 넣고 비동기로 구현하고 싶어요.
AI: asyncio와 함께 사용할 수 있습니다. 타입힌트 포함 예시 코드를 보여드리겠습니다.
사용자: 성능 측정은 어떻게 해요?
AI: NDCG, MRR, Recall@k 등의 메트릭을 사용합니다. RAGAS 프레임워크로 자동 평가도 가능합니다.
사용자: 감사합니다. 나중에 벤치마크 결과 같이 봐주실 수 있나요?
AI: 물론이죠! 언제든 벤치마크 결과를 공유해 주시면 분석해 드리겠습니다."""

tokens_before = count_tokens(long_conversation)
print(f"압축 전 토큰 수: {tokens_before}")

In [ ]:
# LLM을 사용한 대화 압축
compress_prompt = """다음 대화를 간결하게 요약하세요. 반드시 포함할 항목:
1. 사용자 정보 (이름, 직업)
2. 진행 중인 작업
3. 핵심 결정 사항
4. 사용자 선호도
5. 미래 계획

형식: 각 항목을 한 줄로, 불필요한 인사말/감사 표현은 제거."""

response = llm.invoke([
    SystemMessage(content=compress_prompt),
    HumanMessage(content=long_conversation)
])

compressed = response.content
tokens_after = count_tokens(compressed)

print(f"=== 압축 결과 ===")
print(compressed)
print(f"\n=== 토큰 비교 ===")
print(f"압축 전: {tokens_before} 토큰")
print(f"압축 후: {tokens_after} 토큰")
print(f"절약량: {tokens_before - tokens_after} 토큰 ({100 - tokens_after * 100 // tokens_before}% 절약)")

---
## 섹션 8. LangGraph Persistent Memory Agent

LangGraph의 `MemorySaver` checkpointer를 사용하여 세션 간 상태를 유지하는 에이전트를 구현합니다.
`thread_id`로 사용자별 대화 이력을 관리합니다.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Checkpointer 생성 — 인메모리 (프로덕션에서는 PostgresSaver 등 사용)
checkpointer = MemorySaver()

# LLM 노드
def call_model(state: MessagesState):
    """LLM을 호출하여 응답을 생성합니다."""
    system_msg = SystemMessage(
        content="당신은 사용자의 정보를 기억하는 친절한 AI 비서입니다. "
                "이전 대화 내용을 참고하여 개인화된 응답을 제공하세요."
    )
    messages = [system_msg] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}

# 그래프 구성
builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_edge(START, "model")
builder.add_edge("model", END)

# Checkpointer와 함께 컴파일
graph = builder.compile(checkpointer=checkpointer)
print("LangGraph 에이전트 (with Checkpointer) 준비 완료")

In [ ]:
# 세션 1: 자기소개
config = {"configurable": {"thread_id": "user_kim_001"}}

print("=== 세션 1: 자기소개 ===")
response1 = graph.invoke(
    {"messages": [HumanMessage(content="안녕하세요, 저는 박지은이고 데이터 사이언티스트입니다. 주로 시계열 예측 모델을 만들어요.")]},
    config=config
)
print(f"AI: {response1['messages'][-1].content}\n")

# 같은 세션에서 추가 질문
response2 = graph.invoke(
    {"messages": [HumanMessage(content="요즘 Prophet 대신 새로운 모델을 찾고 있어요. 추천해줄 수 있나요?")]},
    config=config
)
print(f"AI: {response2['messages'][-1].content}")

---
## 섹션 9. 세션 간 컨텍스트 유지 데모

같은 `thread_id`를 사용하면 새 세션에서도 이전 대화를 기억합니다.
**핵심**: checkpointer가 세션 간 상태를 자동으로 저장/복원합니다.

In [ ]:
# 세션 2: 이전 대화를 기억하는지 확인 (같은 thread_id 사용)
print("=== 세션 2: 기억 확인 (같은 thread_id) ===")
print(f"thread_id: user_kim_001\n")

response3 = graph.invoke(
    {"messages": [HumanMessage(content="내 이름이랑 직업이 뭐였죠? 그리고 어떤 모델을 찾고 있었는지 기억나요?")]},
    config=config  # 같은 thread_id
)
print(f"AI: {response3['messages'][-1].content}")

In [ ]:
# 다른 thread_id로 접속 — 기억 없음 확인
other_config = {"configurable": {"thread_id": "user_other_999"}}

print("=== 세션 3: 다른 thread_id (기억 없음) ===")
print(f"thread_id: user_other_999\n")

response4 = graph.invoke(
    {"messages": [HumanMessage(content="내 이름이랑 직업이 뭐였죠?")]},
    config=other_config  # 다른 thread_id
)
print(f"AI: {response4['messages'][-1].content}")

print(f"\n--- 결론 ---")
print("같은 thread_id → 이전 대화 기억 O")
print("다른 thread_id → 이전 대화 기억 X")

---
## 섹션 10. 메모리 검색 전략 — Recency + Relevance + Importance

"Generative Agents" (Park et al., 2023) 논문의 메모리 검색 방식을 구현합니다.
3가지 축 (최신성, 관련성, 중요도)의 가중합으로 최적의 기억을 선택합니다.

In [ ]:
from dataclasses import dataclass, field
import numpy as np

@dataclass
class MemoryEntry:
    """메모리 항목."""
    content: str
    memory_type: str  # episodic / semantic / procedural
    importance: float  # 1~10
    timestamp: datetime = field(default_factory=datetime.now)
    embedding: list[float] = field(default_factory=list)


class AdvancedMemoryRetriever:
    """Recency + Relevance + Importance 기반 메모리 검색기."""
    
    def __init__(self, embedder, decay_rate: float = 0.01,
                 w_recency: float = 0.3, w_relevance: float = 0.5, w_importance: float = 0.2):
        self.embedder = embedder
        self.decay_rate = decay_rate
        self.w_recency = w_recency
        self.w_relevance = w_relevance
        self.w_importance = w_importance
        self.memories: list[MemoryEntry] = []
    
    def add(self, content: str, memory_type: str, importance: float,
            timestamp: datetime | None = None):
        """메모리를 추가합니다."""
        emb = self.embedder.encode(content).tolist()
        entry = MemoryEntry(
            content=content,
            memory_type=memory_type,
            importance=importance,
            timestamp=timestamp or datetime.now(),
            embedding=emb
        )
        self.memories.append(entry)
    
    def _cosine_similarity(self, a: list[float], b: list[float]) -> float:
        """코사인 유사도를 계산합니다."""
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    
    def _recency_score(self, memory: MemoryEntry, now: datetime) -> float:
        """최신성 점수: 최근일수록 높음."""
        hours_ago = (now - memory.timestamp).total_seconds() / 3600
        return math.exp(-self.decay_rate * hours_ago)
    
    def _relevance_score(self, memory: MemoryEntry, query_embedding: list[float]) -> float:
        """관련성 점수: 쿼리와 유사할수록 높음."""
        return self._cosine_similarity(query_embedding, memory.embedding)
    
    def _importance_score(self, memory: MemoryEntry) -> float:
        """중요도 점수: 0~1 정규화."""
        return memory.importance / 10.0
    
    def search(self, query: str, k: int = 5) -> list[tuple[MemoryEntry, float, dict]]:
        """가중합 기반 메모리 검색.
        
        Returns:
            List of (memory, total_score, score_breakdown)
        """
        query_emb = self.embedder.encode(query).tolist()
        now = datetime.now()
        
        scored = []
        for mem in self.memories:
            rec = self._recency_score(mem, now)
            rel = self._relevance_score(mem, query_emb)
            imp = self._importance_score(mem)
            total = (self.w_recency * rec + 
                     self.w_relevance * rel + 
                     self.w_importance * imp)
            breakdown = {"recency": rec, "relevance": rel, "importance": imp}
            scored.append((mem, total, breakdown))
        
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]


print("AdvancedMemoryRetriever 클래스 정의 완료")

In [ ]:
# 검색기 생성 및 메모리 추가
retriever = AdvancedMemoryRetriever(embedder)

# 다양한 시점의 메모리 추가 (시간 차이를 시뮬레이션)
now = datetime.now()
test_memories = [
    ("사용자 김철수는 ML 엔지니어이다", "semantic", 9, now - timedelta(hours=1)),
    ("RAG 파이프라인 개선 프로젝트를 진행 중이다", "episodic", 7, now - timedelta(hours=2)),
    ("리랭킹 모델로 Cross-encoder를 선택했다", "episodic", 6, now - timedelta(hours=48)),
    ("Python을 주력 언어로 사용한다", "semantic", 8, now - timedelta(hours=72)),
    ("코드에 타입힌트를 반드시 포함한다", "procedural", 7, now - timedelta(hours=96)),
    ("어제 점심으로 김치찌개를 먹었다", "episodic", 2, now - timedelta(hours=24)),
    ("NDCG, MRR 벤치마크를 실행할 예정이다", "episodic", 6, now - timedelta(hours=3)),
    ("비동기 프로그래밍을 선호한다", "procedural", 6, now - timedelta(hours=120)),
]

for content, mtype, imp, ts in test_memories:
    retriever.add(content, mtype, imp, ts)

print(f"{len(test_memories)}개 메모리 추가 완료\n")

# 검색 테스트
query = "현재 진행 중인 프로젝트와 사용하는 기술은?"
print(f"질문: {query}\n")

results = retriever.search(query, k=5)
print(f"{'순위':>4} {'총점':>6} {'최신성':>6} {'관련성':>6} {'중요도':>6}  내용")
print("-" * 80)
for i, (mem, score, bd) in enumerate(results, 1):
    print(f"{i:>4} {score:>6.3f} {bd['recency']:>6.3f} {bd['relevance']:>6.3f} {bd['importance']:>6.3f}  {mem.content}")

---
## 섹션 11. Langfuse 통합 — 메모리 사용량 추적

메모리 읽기/쓰기 작업을 Langfuse로 추적하여 프로덕션 운영에 활용합니다.

In [ ]:
class TracedMemoryManager:
    """Langfuse 추적이 내장된 메모리 관리자."""
    
    def __init__(self, retriever: AdvancedMemoryRetriever, 
                 kv_memory: KVMemory, langfuse_client: Langfuse):
        self.retriever = retriever
        self.kv = kv_memory
        self.langfuse = langfuse_client
    
    def write_memory(self, content: str, memory_type: str, 
                     importance: float, user_id: str) -> str:
        """메모리를 저장하고 Langfuse에 기록합니다."""
        trace = self.langfuse.trace(
            name="memory_write",
            user_id=user_id,
            metadata={
                "memory_type": memory_type,
                "importance": importance,
                "content_length": len(content),
                "token_count": count_tokens(content),
            }
        )
        
        span = trace.span(name="vector_store_write")
        self.retriever.add(content, memory_type, importance)
        span.end(output={"status": "success"})
        
        trace.update(output={"stored": content[:50]})
        return trace.id
    
    def read_memory(self, query: str, k: int, user_id: str) -> list[dict]:
        """메모리를 검색하고 Langfuse에 기록합니다."""
        trace = self.langfuse.trace(
            name="memory_read",
            user_id=user_id,
            input={"query": query, "k": k}
        )
        
        # Vector DB 검색
        span_vec = trace.span(name="vector_search")
        results = self.retriever.search(query, k=k)
        span_vec.end(output={"results_count": len(results)})
        
        # KV 프로필 조회
        span_kv = trace.span(name="kv_profile_lookup")
        profile = self.kv.to_context_string()
        span_kv.end(output={"profile_tokens": count_tokens(profile)})
        
        # 결과 정리
        formatted = []
        for mem, score, breakdown in results:
            formatted.append({
                "content": mem.content,
                "score": round(score, 3),
                "type": mem.memory_type,
                "breakdown": {k: round(v, 3) for k, v in breakdown.items()},
            })
        
        trace.update(output={
            "top_results": [r["content"][:50] for r in formatted],
            "avg_score": round(sum(r["score"] for r in formatted) / len(formatted), 3) if formatted else 0,
        })
        
        return formatted


print("TracedMemoryManager 클래스 정의 완료")

In [ ]:
# TracedMemoryManager 사용 데모
traced_mgr = TracedMemoryManager(retriever, kv, langfuse)

# 메모리 쓰기 (Langfuse 추적)
print("=== 메모리 쓰기 (Langfuse 추적) ===")
trace_id = traced_mgr.write_memory(
    content="TimesFM 모델을 시계열 예측에 사용해보기로 결정",
    memory_type="episodic",
    importance=8,
    user_id="user_kim_001"
)
print(f"저장 완료 — Langfuse trace_id: {trace_id}")

# 메모리 읽기 (Langfuse 추적)
print(f"\n=== 메모리 읽기 (Langfuse 추적) ===")
results = traced_mgr.read_memory(
    query="시계열 예측 관련 기억",
    k=3,
    user_id="user_kim_001"
)

for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['type']}] (score: {r['score']}) {r['content']}")
    print(f"     breakdown: {r['breakdown']}")

# Langfuse flush
langfuse.flush()
print(f"\nLangfuse 대시보드에서 'memory_write', 'memory_read' 트레이스를 확인하세요.")

---
## 섹션 12. Exercise

### Exercise 1: 통합 메모리 에이전트

**목표**: Short-term (슬라이딩 윈도우) + Long-term (Vector DB + KV) 메모리를 결합한 에이전트 구현

**요구사항**:
1. `IntegratedMemoryAgent` 클래스 구현
2. 대화 5턴마다 자동 압축 (LLM 요약 → Long-term에 저장)
3. 새 세션 시작 시 관련 기억 자동 로드
4. 메모리 검색에 recency + relevance + importance 스코어링 적용
5. Langfuse로 메모리 읽기/쓰기 추적

**평가 기준**: 세션 2에서 세션 1의 정보를 정확히 활용하는가?

In [ ]:
# TODO: Exercise 1 — 통합 메모리 에이전트 구현

class IntegratedMemoryAgent:
    """Short-term + Long-term 통합 메모리 에이전트."""
    
    def __init__(self, llm, retriever, kv_memory, langfuse_client,
                 window_size: int = 5, compress_every: int = 5):
        self.llm = llm
        self.retriever = retriever
        self.kv = kv_memory
        self.langfuse = langfuse_client
        self.window_size = window_size
        self.compress_every = compress_every
        self.short_term: list = []  # 현재 세션 대화
        self.turn_count: int = 0
    
    def chat(self, user_message: str, user_id: str) -> str:
        """사용자 메시지를 처리하고 응답을 반환합니다."""
        # TODO: 1. Long-term에서 관련 기억 검색
        # TODO: 2. KV에서 사용자 프로필 조회
        # TODO: 3. 시스템 프롬프트에 기억 + 프로필 주입
        # TODO: 4. LLM 호출
        # TODO: 5. 응답을 short-term에 추가
        # TODO: 6. compress_every 턴마다 자동 압축
        # TODO: 7. Langfuse 트레이싱
        pass
    
    def _compress_and_store(self, user_id: str):
        """Short-term 대화를 요약하여 Long-term에 저장합니다."""
        # TODO: LLM으로 대화 요약 → retriever에 저장
        pass
    
    def start_new_session(self, user_id: str):
        """새 세션을 시작합니다. 이전 세션의 관련 기억을 로드합니다."""
        # TODO: short_term 초기화 + long-term에서 최근 기억 로드
        pass


# 테스트
# agent = IntegratedMemoryAgent(llm, retriever, kv, langfuse)
# agent.chat("안녕, 나는 김철수야", "user_kim")
# agent.chat("RAG 개선하고 있어", "user_kim")
# ...
# agent.start_new_session("user_kim")
# response = agent.chat("내가 뭘 하고 있었지?", "user_kim")
# print(response)  # RAG 개선 관련 내용을 기억해야 함

print("Exercise 1: IntegratedMemoryAgent 클래스 스텁 정의 완료")
print("TODO 주석을 따라 구현하세요.")

### Exercise 2: 메모리 성능 벤치마크

**목표**: 3가지 저장 전략 (Vector DB / KV / Markdown)의 성능 비교

**측정 항목**:
1. 쓰기 속도: 100개 메모리 저장 시간
2. 읽기 속도: 검색 응답 시간 (ms)
3. 검색 정확도: 관련 메모리 top-5 정밀도
4. 토큰 효율: 전체 메모리의 토큰 수 비교
5. 확장성: 메모리 1000개 시 성능 변화

In [ ]:
# TODO: Exercise 2 — 메모리 성능 벤치마크
import time

def benchmark_write(store_func, memories: list[str], label: str) -> float:
    """쓰기 성능을 측정합니다."""
    # TODO: 100개 메모리 쓰기 시간 측정
    start = time.time()
    # TODO: store_func으로 각 메모리 저장
    elapsed = time.time() - start
    print(f"[{label}] {len(memories)}개 쓰기: {elapsed:.3f}초")
    return elapsed


def benchmark_read(search_func, queries: list[str], label: str) -> float:
    """읽기 성능을 측정합니다."""
    # TODO: 쿼리별 검색 시간 측정
    start = time.time()
    # TODO: search_func으로 각 쿼리 검색
    elapsed = time.time() - start
    avg_ms = elapsed / len(queries) * 1000
    print(f"[{label}] {len(queries)}개 쿼리 평균: {avg_ms:.1f}ms")
    return avg_ms


# TODO: 테스트 데이터 생성
# test_memories = [f"기억 {i}: 테스트 내용 ..." for i in range(100)]
# test_queries = ["직업", "프로젝트", "선호도", "기술 스택", "최근 작업"]

# TODO: 각 저장소별 벤치마크 실행
# benchmark_write(vectordb_store, test_memories, "VectorDB")
# benchmark_write(kv_store, test_memories, "KV Store")
# benchmark_write(md_store, test_memories, "Markdown")

# TODO: 결과 테이블 출력
# | 저장소    | 쓰기 (100개) | 읽기 (평균) | 토큰 수 |
# |----------|-------------|-----------|--------|
# | VectorDB | ???초       | ???ms     | ???    |
# | KV Store | ???초       | ???ms     | ???    |
# | Markdown | ???초       | ???ms     | ???    |

print("Exercise 2: 벤치마크 스텁 정의 완료")
print("TODO 주석을 따라 구현하세요.")

---
## 마무리

### 오늘 배운 것

- **Short-term vs Long-term 메모리**를 구분하여 토큰을 효율적으로 사용하는 방법
- **3가지 저장 전략** (Vector DB / KV Store / Markdown)의 장단점과 구현
- **메모리 압축**으로 오래된 대화를 요약하여 토큰 80%+ 절약
- **LangGraph Checkpointer**로 세션 간 상태를 자동으로 유지
- **Recency + Relevance + Importance** 가중합 기반 메모리 검색
- **Langfuse**로 메모리 읽기/쓰기를 추적하여 프로덕션 운영

### 다음 에피소드

**EP21**: 에이전트가 실수했을 때 — 자동 복구와 폴백 전략

### 참고 자료

- [Generative Agents: Interactive Simulacra of Human Behavior](https://arxiv.org/abs/2304.03442) — Park et al., 2023
- [LangGraph Persistence](https://langchain-ai.github.io/langgraph/concepts/persistence/)
- [ChromaDB Documentation](https://docs.trychroma.com/)
- [Langfuse Tracing](https://langfuse.com/docs/tracing)

> 전체 코드는 GitHub 레포에서, 심화 내용은 커뮤니티에서